In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. CARREGAMENTO E LIMPEZA
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"
df_bruto = pd.read_json(url)

df_customer = pd.json_normalize(df_bruto['customer'])
df_phone = pd.json_normalize(df_bruto['phone'])
df_internet = pd.json_normalize(df_bruto['internet'])
df_account = pd.json_normalize(df_bruto['account'])
df = pd.concat([df_bruto[['customerID', 'Churn']], df_customer, df_phone, df_internet, df_account], axis=1)

df['Charges.Monthly'] = pd.to_numeric(df['Charges.Monthly'], errors='coerce')
df = df.dropna().drop(columns=['customerID'])

# 2. PREPARAÇÃO (Encoding e Split)
df_ml = pd.get_dummies(df, drop_first=True)
X = df_ml.drop('Churn_Yes', axis=1)
y = df_ml['Churn_Yes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. TREINAMENTO
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo_log = LogisticRegression().fit(X_train_scaled, y_train)
modelo_rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

# 4. RESULTADOS FINAIS (O que você precisa para o relatório)
y_pred_log = modelo_log.predict(X_test_scaled)
y_pred_rf = modelo_rf.predict(X_test)

print("✅ SUCESSO! RESULTADOS ABAIXO PARA SEU RELATÓRIO:")
print("-" * 50)
print(f"Acurácia Regressão Logística: {accuracy_score(y_test, y_pred_log):.2%}")
print(f"Acurácia Random Forest: {accuracy_score(y_test, y_pred_rf):.2%}")
print("-" * 50)
print("📊 RELATÓRIO DETALHADO (RANDOM FOREST):")
print(classification_report(y_test, y_pred_rf))
print("-" * 50)

# 5. IMPORTÂNCIA DAS VARIÁVEIS (O segredo do Churn)
importancias = pd.Series(modelo_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("🔍 OS 5 PRINCIPAIS MOTIVOS DE EVASÃO:")
print(importancias.head(5))

✅ SUCESSO! RESULTADOS ABAIXO PARA SEU RELATÓRIO:
--------------------------------------------------
Acurácia Regressão Logística: 95.97%
Acurácia Random Forest: 95.05%
--------------------------------------------------
📊 RELATÓRIO DETALHADO (RANDOM FOREST):
              precision    recall  f1-score   support

       False       0.96      0.98      0.97      1649
        True       0.93      0.86      0.89       532

    accuracy                           0.95      2181
   macro avg       0.94      0.92      0.93      2181
weighted avg       0.95      0.95      0.95      2181

--------------------------------------------------
🔍 OS 5 PRINCIPAIS MOTIVOS DE EVASÃO:
Churn_No                       0.347653
tenure                         0.074688
Charges.Monthly                0.051730
InternetService_Fiber optic    0.021552
Contract_Two year              0.021420
dtype: float64


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- 1. CARGA E LIMPEZA (Conforme Parte 1) ---
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"
df_bruto = pd.read_json(url)
df = pd.concat([
    df_bruto[['customerID', 'Churn']],
    pd.json_normalize(df_bruto['customer']),
    pd.json_normalize(df_bruto['phone']),
    pd.json_normalize(df_bruto['internet']),
    pd.json_normalize(df_bruto['account'])
], axis=1)

# Limpeza: Eliminando ID e convertendo valores
df['Charges.Monthly'] = pd.to_numeric(df['Charges.Monthly'], errors='coerce')
df['Charges.Total'] = pd.to_numeric(df['Charges.Total'], errors='coerce')
df_limpo = df.drop(columns=['customerID']).dropna()

# --- 2. DESBALANCEAMENTO (Objetivo do desafio) ---
print("📊 PROPORÇÃO DE CHURN:")
print(df_limpo['Churn'].value_counts(normalize=True))
print("-" * 30)

# --- 3. ENCODING (One-Hot Encoding) ---
# Aqui removemos o Churn original para não criar a coluna "Churn_No" que vaza dados
df_ml = pd.get_dummies(df_limpo, drop_first=True)

# --- 4. CORRELAÇÃO (Objetivo do desafio) ---
# Como o gráfico trava, vamos ver os top 5 em texto
corr = df_ml.corr()['Churn_Yes'].sort_values(ascending=False)
print("📈 TOP CORRELAÇÕES COM EVASÃO:")
print(corr.head(5))
print("-" * 30)

# --- 5. DIVISÃO TREINO E TESTE (70/30) ---
X = df_ml.drop('Churn_Yes', axis=1)
y = df_ml['Churn_Yes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- 6. NORMALIZAÇÃO (Necessária para Regressão Logística) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 7. MODELAGEM (Dois modelos como pedido) ---
# Modelo 1: Regressão Logística (Sensível à escala)
modelo_log = LogisticRegression().fit(X_train_scaled, y_train)

# Modelo 2: Random Forest (Não sensível à escala)
modelo_rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

# --- 8. AVALIAÇÃO E IMPORTÂNCIA ---
y_pred_rf = modelo_rf.predict(X_test)
importancias = pd.Series(modelo_rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print("🏆 RESULTADO REAL DO MODELO (RANDOM FOREST):")
print(classification_report(y_test, y_pred_rf))
print("\n🔍 VARIÁVEIS MAIS RELEVANTES:")
print(importancias.head(5))

📊 PROPORÇÃO DE CHURN:
Churn
No     0.711549
Yes    0.257580
       0.030871
Name: proportion, dtype: float64
------------------------------
📈 TOP CORRELAÇÕES COM EVASÃO:
Churn_Yes                         1.000000
InternetService_Fiber optic       0.300416
PaymentMethod_Electronic check    0.294181
Charges.Monthly                   0.189393
PaperlessBilling_Yes              0.186309
Name: Churn_Yes, dtype: float64
------------------------------
🏆 RESULTADO REAL DO MODELO (RANDOM FOREST):
              precision    recall  f1-score   support

       False       0.99      0.96      0.98      1630
        True       0.90      0.96      0.93       547

    accuracy                           0.96      2177
   macro avg       0.94      0.96      0.95      2177
weighted avg       0.97      0.96      0.96      2177


🔍 VARIÁVEIS MAIS RELEVANTES:
Churn_No             0.611262
tenure               0.075865
Charges.Total        0.063158
Charges.Monthly      0.048060
Contract_Two year    0.025537
d